# Under the hood · Gradient boosting from scratch

**Reference notebook.** Not a meeting — this is where Meeting 17 sends you when
"sequential error correction" is not enough of an explanation.

By the end you will have written gradient boosting in six lines and checked
that it produces *bit-for-bit identical* predictions to
`sklearn.ensemble.GradientBoostingRegressor`.

Boosting is the most important practical method in this course — it is what
usually wins on tabular data, and it is what the foundation model in Meeting 18
has to beat. It deserves more than an analogy.

In [ ]:
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])

if found:                       # you are inside (or just outside) your clone
    sys.path.insert(0, str(found[0] / "course"))
else:                           # Colab, or a copy saved outside the clone
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

from stat764 import load

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")

ames = load("ames.csv")
NUMERIC = ["Gr_Liv_Area", "Lot_Area", "Year_Built", "Overall_Qual",
           "Total_Bsmt_SF", "Garage_Cars"]

X = ames[NUMERIC].fillna(ames[NUMERIC].median()).to_numpy(float)
y = ames["SalePrice"].to_numpy(float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=764)

LEARNING_RATE = 0.1
DEPTH = 3

## 1. The idea

> Fit a model. Look at what it got wrong. **Fit a second model to the mistakes.**
> Add a small fraction of it to the first. Repeat.

That is the whole algorithm. Everything else is bookkeeping.

Contrast it with the random forest from Meeting 16: a forest fits hundreds of
trees **independently** on bootstrap samples and averages them, which reduces
variance. Boosting fits each tree **on the errors of everything before it**,
which reduces bias. Forests are a committee voting in parallel; boosting is one
apprentice repeatedly patching its own work.

## 2. Round by round, out loud

Start with the least imaginative model available: predict the mean for
everybody. Call the current prediction `F`.

In [ ]:
F = np.full(len(y_train), y_train.mean())
print(f"Round 0: predict ${F[0]:,.0f} for every house")
print(f"         training R-squared {r2_score(y_train, F):.4f}")

The **residual** is what this model got wrong, house by house. It is the part
of the answer we still owe.

In [ ]:
residual = y_train - F
print(pd.DataFrame({
    "actual": y_train[:5],
    "predicted": F[:5],
    "residual (still owed)": residual[:5],
}).round(0).to_string(index=False))

Now fit a small tree — **to the residuals, not to the prices.** This tree's
only job is to explain the leftovers.

Then add a *fraction* of its prediction to `F`. The fraction is the learning
rate, and taking small steps rather than the full correction is most of why
boosting works.

In [ ]:
tree1 = DecisionTreeRegressor(max_depth=DEPTH, random_state=0).fit(X_train, residual)
F = F + LEARNING_RATE * tree1.predict(X_train)

print(f"Round 1: training R-squared {r2_score(y_train, F):.4f}")
print(f"         mean |residual| went from ${np.abs(y_train - y_train.mean()).mean():,.0f} "
      f"to ${np.abs(y_train - F).mean():,.0f}")

Again. Same three steps: compute what is still owed, fit a tree to it, add a
tenth of that tree.

In [ ]:
for round_number in range(2, 6):
    residual = y_train - F
    tree = DecisionTreeRegressor(max_depth=DEPTH, random_state=0).fit(X_train, residual)
    F = F + LEARNING_RATE * tree.predict(X_train)
    print(f"Round {round_number}: training R-squared {r2_score(y_train, F):.4f}   "
          f"mean |residual| ${np.abs(y_train - F).mean():,.0f}")

Each tree is nearly useless on its own — depth 3, fit to leftovers, scaled down
to a tenth. Two hundred of them in sequence are formidable.

## 3. The whole algorithm

Six lines. This *is* gradient boosting for squared-error loss; there is nothing
withheld.

In [ ]:
def fit_boosting(X, y, n_rounds, learning_rate=0.1, depth=3):
    F = np.full(len(y), y.mean())
    trees = []
    for _ in range(n_rounds):
        tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X, y - F)
        F = F + learning_rate * tree.predict(X)
        trees.append(tree)
    return y.mean(), trees


def predict_boosting(model, X_new, learning_rate=0.1):
    base, trees = model
    out = np.full(len(X_new), base)
    for tree in trees:
        out = out + learning_rate * tree.predict(X_new)
    return out


model = fit_boosting(X_train, y_train, n_rounds=20)
mine = predict_boosting(model, X_test)
print(f"20 rounds, from scratch: test R-squared {r2_score(y_test, mine):.6f}")

## 4. Is that really what scikit-learn does?

Run the library version with the same settings and compare predictions
house by house.

In [ ]:
sk = GradientBoostingRegressor(n_estimators=20, learning_rate=LEARNING_RATE,
                               max_depth=DEPTH, random_state=0).fit(X_train, y_train)
theirs = sk.predict(X_test)

print(f"  from scratch : R-squared {r2_score(y_test, mine):.6f}")
print(f"  scikit-learn : R-squared {r2_score(y_test, theirs):.6f}")
print(f"  largest disagreement on any single house: ${np.abs(mine - theirs).max():.10f}")

The largest disagreement across all 733 test houses is **one ten-billionth of a
dollar** — about 1e-10 on values near $180,000, which is fifteen significant
figures of agreement. That is the residue of adding twenty numbers in a
different order, not a difference in the algorithm.

The six-line function is not a simplified illustration of gradient boosting.
It is gradient boosting.

## 5. Where it stops matching, and why that is worth knowing

Run it longer and the two implementations drift apart.

In [ ]:
rows = []
for n in [1, 5, 20, 50, 100, 200]:
    a = predict_boosting(fit_boosting(X_train, y_train, n), X_test)
    b = GradientBoostingRegressor(n_estimators=n, learning_rate=LEARNING_RATE,
                                  max_depth=DEPTH, random_state=0).fit(X_train, y_train).predict(X_test)
    rows.append({"rounds": n, "max disagreement": np.abs(a - b).max(),
                 "R2 scratch": r2_score(y_test, a), "R2 sklearn": r2_score(y_test, b)})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:,.6f}"))

Exact through 20 rounds, then it separates.

Nothing is broken. By round 50 the residuals are small and many candidate
splits are near-ties; the two implementations break those ties differently, and
because every later tree is fit to the output of every earlier one, a
hair's-width difference compounds.

This is worth sitting with, because it is a property of the method rather than
a bug: **boosting is sequentially dependent, so it amplifies small numerical
differences.** A forest would not do this — its trees are independent. When you
cannot reproduce someone's boosted model exactly, this is usually why.

## 6. Why is it called *gradient* boosting?

We never computed a gradient. We fit trees to residuals.

For squared-error loss those are the same thing. With
$L(y, F) = \frac{1}{2}(y - F)^2$,

$$-\frac{\partial L}{\partial F} = y - F = \text{residual}$$

So "fit the next tree to the residuals" is really **"fit the next tree to the
negative gradient of the loss,"** and each round is one step of gradient
descent — not in parameter space, but in the space of functions. The learning
rate is the step size, exactly as it would be in any other gradient descent.

That reframing is what generalizes the method. Change the loss and the same
algorithm works; only the thing you fit trees to changes:

| loss | what each tree gets fit to |
|---|---|
| squared error | $y - F$, the residual |
| absolute error | $\text{sign}(y - F)$ |
| log-loss (classification) | $y - p$, observed minus predicted probability |

Classification boosting is the same six lines with the third row substituted in.

## 7. The three knobs are not independent

`learning_rate`, `n_estimators`, and `max_depth` are usually presented as three
things to tune. The first two are really one thing: **how much total correction
gets applied**, split into step size and number of steps.

Halve the learning rate and you need roughly twice as many rounds for the same
fit. Which is why tuning `n_estimators` on its own is close to meaningless.

In [ ]:
rows = []
for lr, depth in [(0.1, 3), (0.1, 6), (0.5, 6), (1.0, 8)]:
    m = GradientBoostingRegressor(n_estimators=1500, learning_rate=lr,
                                  max_depth=depth, random_state=0).fit(X_train, y_train)
    test_curve = np.array([r2_score(y_test, p) for p in m.staged_predict(X_test)])
    train_curve = np.array([r2_score(y_train, p) for p in m.staged_predict(X_train)])
    best = int(test_curve.argmax())
    rows.append({"learning rate": lr, "depth": depth,
                 "best round": best + 1, "best test R2": test_curve[best],
                 "train R2 @1500": train_curve[-1], "test R2 @1500": test_curve[-1]})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

Read the last row. At a learning rate of 1.0 with depth 8, **the best number of
boosting rounds is one.** Every tree after the first makes the model worse. The
steps are so large that the procedure overshoots immediately.

Read the first row too: the best model appears around round 191, and by round
1500 training R-squared has climbed to 0.989 while test R-squared has *fallen*
to 0.893. That is the Meeting 3 flexibility curve, back again, with the number
of rounds playing the role tree depth played then.

**`n_estimators` is not a capacity setting you turn up until satisfied. It is a
regularization parameter, and more is not better.**

In [ ]:
m = GradientBoostingRegressor(n_estimators=1500, learning_rate=0.1, max_depth=3,
                              random_state=0).fit(X_train, y_train)
train_curve = np.array([r2_score(y_train, p) for p in m.staged_predict(X_train)])
test_curve = np.array([r2_score(y_test, p) for p in m.staged_predict(X_test)])
best = int(test_curve.argmax())

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(train_curve, label="training data")
ax.plot(test_curve, label="unseen data")
ax.axvline(best, color="gray", linestyle=":", linewidth=1)
ax.annotate(f"best at round {best + 1}", xy=(best, test_curve[best]),
            xytext=(best + 220, test_curve[best] - 0.07),
            arrowprops=dict(arrowstyle="->", color="gray"), fontsize=9, color="gray")
ax.set_xlabel("boosting rounds")
ax.set_ylabel("R-squared")
ax.set_title("More rounds is not more better")
ax.set_ylim(0.6, 1.02)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. What boosting costs

It usually wins on tabular data. It is not free.

| | |
|---|---|
| **Tuning burden** | Three coupled knobs, and the defaults are rarely right. A forest works out of the box; boosting does not. |
| **Sequential** | Trees cannot be fit in parallel, and small numerical differences compound (section 5). |
| **Overfits with enough rounds** | Unlike a random forest, which mostly plateaus. `n_estimators` must be chosen, not maximized. |
| **No uncertainty** | It returns a number with no interval. Meeting 13 is where we fix that, and conformal prediction works on top of a boosted model without modification. |
| **Extrapolation** | Trees predict a constant outside the range they were trained on. A boosted model asked about a 6,000 sq ft house when it has only seen 4,000 will not extrapolate — it will return the edge value. |

The last row is worth remembering at capstone time.

---

## Check yourself

1. Modify `fit_boosting` to take a validation set and stop when validation error
   stops improving. How many rounds does it choose?
2. Change the loss to absolute error — fit each tree to `np.sign(y - F)` instead
   of `y - F`. Does it handle the expensive-house outliers differently?
3. Set `learning_rate=1.0` in `fit_boosting` and watch what happens to the
   training residuals in the first five rounds. Explain it.